# 1.1 First NGSolve example

Let us solve the Poisson problem of finding $u$ satisfying 

$$
\begin{aligned}
-\Delta u & = f && \text { in  the unit square},
\\
u & = 0 && \text{ on the bottom and right parts of the boundary},
\\
\frac{\partial u }{\partial n } & = 0 
&& \text{ on the remaining  boundary parts}.
\end{aligned}
$$

## Quick steps to solution:

#### 1. Import NGSolve and Netgen Python modules:

In [2]:
import sys
print(sys.executable)
from ngsolve import *
from ngsolve.webgui import Draw

F:\changeworld\HPMCalc\venv\Scripts\python.exe


#### 2. Generate an unstructured mesh

In [3]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))
mesh.nv, mesh.ne   # number of vertices & elements 

(39, 56)

Here we prescribed a maximal mesh-size of 0.2 using the `maxh` flag. 

In [4]:
Draw(mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

#### 3. Declare a finite element space:

In [7]:
fes = H1(mesh, order=2, dirichlet="bottom|right")
fes.ndof  # number of unknowns in this space

133

Python's help system displays further documentation.

In [8]:
help(fes)

Help on H1 in module ngsolve.comp object:

class H1(FESpace)
 |  An H1-conforming finite element space.
 |  
 |  The H1 finite element space consists of continuous and
 |  element-wise polynomial functions. It uses a hierarchical (=modal)
 |  basis built from integrated Legendre polynomials on tensor-product elements,
 |  and Jaboci polynomials on simplicial elements. 
 |  
 |  Boundary values are well defined. The function can be used directly on the
 |  boundary, using the trace operator is optional.
 |  
 |  The H1 space supports variable order, which can be set individually for edges, 
 |  faces and cells. 
 |  
 |  Internal degrees of freedom are declared as local dofs and are eliminated 
 |  if static condensation is on.
 |  
 |  The wirebasket consists of all vertex dofs. Optionally, one can include the 
 |  first (the quadratic bubble) edge basis function, or all edge basis functions
 |  into the wirebasket.
 |  
 |  Keyword arguments can be:
 |  
 |  order: int = 1
 |    order

#### 4. Declare test function, trial function, and grid function 

* Test and trial function are symbolic objects - called `ProxyFunctions` -  that help you construct bilinear forms (and have no space to hold solutions). 

* `GridFunctions`, on the other hand, represent functions in the finite element space and contains memory to hold coefficient vectors.

In [9]:
u = fes.TrialFunction()  # symbolic object
v = fes.TestFunction()   # symbolic object
gfu = GridFunction(fes)  # solution 

Alternately, you can get both the trial and test variables at once:

In [10]:
u, v = fes.TnT()

#### 5. Define and assemble linear and bilinear forms:

In [11]:
a = BilinearForm(fes)
a += grad(u)*grad(v)*dx
a.Assemble()

f = LinearForm(fes)
f += x*v*dx
f.Assemble();

Alternately, we can do one-liners: 

In [52]:
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
# 此处的f并非原微分方程的右侧项（源项）。下面的代码表示源项为x
f = LinearForm( 1e-20*v*dx).Assemble()#LinearForm(x*v*dx).Assemble()

You can examine the linear system in more detail:

In [22]:
# print(f.vec)
(f.vec.size)

133

In [14]:
print(a.mat)

Row 0:   0: 1   4: -0.5   19: -0.5   39: -0.0833333   40: -0.0833333   50: 0.166667
Row 1:   1: 0.828732   7: -0.195774   8: -0.197948   23: -0.43501   41: -0.0360819   42: -0.0364197   43: -0.0656203   59: 0.0687109   61: 0.069411
Row 2:   2: 1   11: -0.5   12: -0.5   44: -0.0833333   45: -0.0833333   69: 0.166667
Row 3:   3: 0.834885   15: -0.164012   16: -0.157951   30: -0.512921   46: -0.0431063   47: -0.0423806   48: -0.0536605   81: 0.0704417   83: 0.0687058
Row 4:   0: -0.5   4: 1.89671   5: -0.546735   19: -0.180532   20: -0.66944   39: 5.83717e-17   40: 0.0833333   49: -0.0544706   50: -0.140436   51: -0.121211   53: 0.145593   91: 0.0871914
Row 5:   4: -0.546735   5: 1.88547   6: -0.352441   20: -0.196284   21: -0.790007   49: -0.0136061   51: 0.104729   52: -0.0481159   53: -0.174674   54: -0.0778481   56: 0.106856   92: 0.10266
Row 6:   5: -0.352441   6: 1.75375   7: -0.265716   21: -0.432048   22: -0.703548   52: -0.0385388   54: 0.097279   55: -0.0537377   56: -0.12226   

#### 6. Solve the system:

In [56]:
gfu.vec.data = \
    a.mat.Inverse(freedofs=fes.FreeDofs()) * f.vec
Draw(gfu,#min = 0, max = 1
    )

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

In [62]:
a = BilinearForm(-div(grad(u))*(v)*dx).Assemble()
# 此处的f并非原微分方程的右侧项（源项）。下面的代码表示源项为x
f = LinearForm( 1e-20*v*dx).Assemble()#LinearForm(x*v*dx).Assemble()

Exception: cannot form div

In [60]:
help(grad)

Help on function grad in module ngsolve.utils:

grad(func)



In [45]:
fes.FreeDofs()

The Dirichlet boundary condition constrains some degrees of freedom. The argument `fes.FreeDofs()` indicates that only the remaining "free" degrees of freedom should participate in the linear solve.

You can examine the coefficient vector of solution if needed:

In [46]:
print(gfu.vec)

       0
       0
       0
 0.0923179
       0
       0
       0
       0
       0
       0
       0
       0
 0.0578988
 0.0863391
 0.0954151
 0.0945036
 0.0888226
 0.0780489
 0.0595559
 0.0331334
 0.043177
 0.0384731
 0.0312766
 0.0180508
 0.0379191
 0.0426386
 0.0472136
 0.0758445
 0.0892927
 0.0932644
 0.0919339
  0.0863
 0.0722526
 0.0576401
 0.0666805
 0.0710587
 0.0846294
 0.0618883
 0.0766818
       0
 -0.00563337
       0
       0
 0.0203536
       0
 -0.0350728
 0.00280938
 -0.00728756
 -0.00137235
       0
 -0.00966674
 -0.0121644
       0
 -0.0238901
 -0.0131488
       0
 -0.0161892
 -0.0200118
 -0.00657505
 -0.0212905
       0
 -0.0255408
 -0.00925383
       0
 -0.0375247
 -0.0131487
       0
 -0.0261275
 -0.0183435
 -0.0333457
 -0.0247246
 -0.0242762
 -0.00473494
 -0.0120835
 -0.0145349
 -0.00640155
 -0.0108854
 -0.00549118
 -0.00937973
 -0.00705122
 -0.0052206
 -0.00304656
 -0.00758425
 0.00107111
 0.00110145
 -0.00824163
 0.000388213
 0.00186718
 -0.0082305
 -0.00056295

You can see the zeros coming from the zero boundary conditions.

## Ways to interact with NGSolve

* A jupyter notebook (like this one) gives you one way to interact with NGSolve. When you have a complex sequence of tasks to perform, the notebook may not be adequate.


* You can write an entire python module in a text editor and call python on the command line. (A script of the above is provided in `poisson.py`.)
    ```
    python3 poisson.py
    ```
  
* If you want the Netgen GUI, then use `netgen` on the command line:
    ```
    netgen poisson.py
    ```
  You can then ask for a python shell from the GUI's menu options (`Solve -> Python shell`).
  